In [ ]:
'''
prompt:
Gere codigo python que se conecta na API do Gemini e submete um prompt.
'''

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import time
from transformers import BitsAndBytesConfig

def load_model(model_name):
    # Enable quantization using bitsandbytes (e.g., 4-bit)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,  # Enable 4-bit quantization
        bnb_4bit_compute_dtype=torch.float16,  # Use float16 for computations
        bnb_4bit_use_double_quant=True,  # Use double quantization for memory efficiency
        bnb_4bit_quant_type="nf4"  # Use normalized float4 for better accuracy
    )

    start_time = time.time()

    # Load tokenizer and quantized model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",  # Automatically map the model to available devices
    )

    # Create the pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        #device=0  # Use GPU (if available) for inference
    )

    end_time = time.time()
    elapsed_time = end_time - start_time

    print('MODEL LOADED!', elapsed_time, "secs")
    return pipe

KeyboardInterrupt: 

In [ ]:
pip install google-generativeai


In [ ]:
with open("api_key.txt", "r") as f:
    API_KEY = f.read().strip()


In [ ]:
from google.generativeai import list_models
import google.generativeai as genai

genai.configure(api_key=API_KEY)

for m in list_models():
    print(m.name)

In [ ]:
import google.generativeai as genai



model_name = "gemini-2.5-flash"

# Configura a API
genai.configure(api_key=API_KEY)

# Cria o modelo
model = genai.GenerativeModel(model_name)

# Submete o prompt
prompt = "Explique a teoria da relatividade em termos simples"
#response = model.generate_content(prompt)

# Exibe a resposta
#print(response.text)


In [ ]:
import pandas as pd

# Caminho para o arquivo CSV

language = 'portuguese'
caminho_csv = f"../data/processed/oab_with_firac_{language}.csv"  # ou "oab.csv" se estiver na raiz do projeto

# Lê o CSV em um DataFrame
oab_questions_df = pd.read_csv(caminho_csv)

# Visualiza as primeiras linhas
oab_questions_df.head()


In [ ]:
import pandas as pd

# Caminho para o arquivo CSV
caminho_csv = "../data/processed/oab_guess_the_rule.csv"  # ou "oab.csv" se estiver na raiz do projeto

# Lê o CSV em um DataFrame
oab_guess_the_rule_df = pd.read_csv(caminho_csv)

# Visualiza as primeiras linhas
oab_guess_the_rule_df.head()

In [ ]:
import pandas as pd

# Caminho para o arquivo CSV
caminho_csv = "../data/processed/oab_guess_the_rule.csv"  # ou "oab.csv" se estiver na raiz do projeto
oab_guess_the_rule_questions_df = pd.read_csv(caminho_csv)

language = 'portuguese'
caminho_csv = f"../data/processed/oab_with_firac_{language}.csv"  # ou "oab.csv" se estiver na raiz do projeto
oab_questions_df = pd.read_csv(caminho_csv)


In [ ]:
# Caminho para o arquivo
caminho_prompt_question = f"prompts/solve_question_prompt_{language}.txt"
caminho_prompt_guess_rule = "prompts/solve_question_rule_prompt.txt"

# Lê o conteúdo do arquivo como string
with open(caminho_prompt_question, "r", encoding="utf-8") as arquivo:
    prompt_text_question = arquivo.read()


# Lê o conteúdo do arquivo como string
with open(caminho_prompt_guess_rule, "r", encoding="utf-8") as arquivo:
    prompt_guess_rule = arquivo.read()

In [ ]:
import re

def parse_answer(response):
    """
    Extrai a alternativa da resposta a partir de um texto que contenha algo como
    'Resposta: B', 'resposta: [B]' ou 'cevap: C'.
    Retorna 'A', 'B', 'C' ou 'D'. Se não encontrar, retorna None.
    """
    padrao = r"(resposta|cevap):\s*\[?([ABCD])\]?"
    match = re.search(padrao, response, flags=re.IGNORECASE)
    if match:
        return match.group(2).upper()
    return None

# Testes
assert parse_answer("resposta: C") == "C"
assert parse_answer("resposta: [B]") == "B"
assert parse_answer("cevap: C") == "C"


In [ ]:
import json

def normalize_json_like_string(s):
    # Primeiro: trocar aspas simples nas chaves por aspas duplas
    s = re.sub(r"'(\w+)'\s*:", r'"\1":', s)
    # Segundo: trocar aspas simples nos valores por aspas duplas,
    # mas sem mexer nas aspas que já são duplas
    s = re.sub(r":\s*'([^']*)'", r': "\1"', s)
    # Terceiro: trocar aspas simples dentro de listas ou dicionários
    s = re.sub(r"\[\s*'([^']*)'\s*\]", r'["\1"]', s)
    return s

def get_leis(rules, conteudo):
   if conteudo == "nomes":
      rules = rules.replace("\"b\"", "b").replace("\'nı", "ni")
      print('rules:', rules)
      rules = normalize_json_like_string(rules)

      extract_lei = lambda o: ([o["lei"]] if isinstance(o, dict) and "lei" in o else []) + sum((extract_lei(v) for v in (o.values() if isinstance(o, dict) else o if isinstance(o, list) else [])), [])
      return ",".join(map(str, extract_lei(json.loads(rules))))
   else:
      return rules


get_leis("""{"constituicao_federal": [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [{"lei": "Art. 852-A", "conteudo": "Os diss\u00eddios individuais cujo valor n\u00e3o exceda a quarenta vezes o sal\u00e1rio m\u00ednimo vigente na data do ajuizamento da reclama\u00e7\u00e3o ficam submetidos ao procedimento sumar\u00edssimo."}, {"lei": "Art. 852-A, par\u00e1grafo \u00fanico", "conteudo": "Est\u00e3o exclu\u00eddas do procedimento sumar\u00edssimo as demandas em que \u00e9 parte a Administra\u00e7\u00e3o P\u00fablica direta, aut\u00e1rquica e fundacional."}], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [], "codigo_tributario_nacional": [], "principio_geral_do_direito": [{"lei": "N/A", "conteudo": "A escolha do rito processual \u00e9 mat\u00e9ria de ordem p\u00fablica, n\u00e3o cabendo \u00e0 parte a op\u00e7\u00e3o."}], "outros": [{"lei": "N/A", "conteudo": "A exist\u00eancia de litiscons\u00f3rcio passivo, por si s\u00f3, n\u00e3o obriga a ado\u00e7\u00e3o do rito ordin\u00e1rio."}, {"lei": "N/A", "conteudo": "Quando a Fazenda P\u00fablica figurar no polo passivo, a a\u00e7\u00e3o deve obrigatoriamente seguir o rito ordin\u00e1rio, independentemente do valor da causa."}]}""", "nomes")

get_leis("""{"constituicao_federal": {"lei": ["Art. 5\u00ba, LIX, da Constitui\u00e7\u00e3o Federal de 1988"], "conteudo": ["ser\u00e1 admitida a\u00e7\u00e3o privada nos crimes de a\u00e7\u00e3o p\u00fablica, se esta n\u00e3o for intentada no prazo legal"]}, "codigo_penal": {"lei": [], "conteudo": []}, "codigo_processo_penal": {"lei": ["Art. 29 do C\u00f3digo de Processo Penal"], "conteudo": ["Ser\u00e1 admitida a\u00e7\u00e3o privada nos crimes de a\u00e7\u00e3o p\u00fablica, se esta n\u00e3o for intentada no prazo legal, cabendo ao Minist\u00e9rio P\u00fablico aditar a queixa, repudi\u00e1-la e oferecer den\u00fancia substitutiva, intervir em todos os termos do processo, fornecer elementos de prova, interpor recurso e, a todo tempo, no caso de neglig\u00eancia do querelante, retomar a a\u00e7\u00e3o como parte principal."]}, "clt": {"lei": [], "conteudo": []}, "codigo_civil": {"lei": [], "conteudo": []}, "codigo_processo_civil": {"lei": [], "conteudo": []}, "codigo_direito_consumidor": {"lei": [], "conteudo": []}, "estatuto_crianca_adolescente": {"lei": [], "conteudo": []}, "sumula_stj": {"lei": [], "conteudo": []}, "estatuto_oab": {"lei": [], "conteudo": []}, "codigo_tributario_nacional": {"lei": [], "conteudo": []}, "principio_geral_do_direito": {"lei": [], "conteudo": []}, "outros": {"lei": [], "conteudo": []}}""", "nomes")

get_leis(""" {"constituicao_federal": [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [], "codigo_tributario_nacional": [{"lei": "Art. 132 do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A pessoa jur\u00eddica de direito privado que resultar de fus\u00e3o, transforma\u00e7\u00e3o ou incorpora\u00e7\u00e3o de outra ou em outra \u00e9 respons\u00e1vel pelos tributos devidos at\u00e9 \u00e0 data do ato pelas pessoas jur\u00eddicas de direito privado fusionadas, transformadas ou incorporadas."}, {"lei": "Art. 133, inciso I do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 integral e isolada, no caso de o alienante cessar a explora\u00e7\u00e3o da atividade."}, {"lei": "Art. 133, inciso II do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 subsidi\u00e1ria, se o alienante prosseguir na explora\u00e7\u00e3o da atividade ou, dentro de seis meses contados da data da aliena\u00e7\u00e3o, iniciar nova atividade no mesmo ou em outro ramo de com\u00e9rcio, ind\u00fastria ou profiss\u00e3o."}], "principio_geral_do_direito": [], "outros": []}""", "nomes")


get_leis(""" {"constituicao_federal": [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [], "codigo_tributario_nacional": [{"lei": "Art. 132 do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A pessoa jur\u00eddica de direito privado que resultar de fus\u00e3o, transforma\u00e7\u00e3o ou incorpora\u00e7\u00e3o de outra ou em outra \u00e9 respons\u00e1vel pelos tributos devidos at\u00e9 \u00e0 data do ato pelas pessoas jur\u00eddicas de direito privado fusionadas, transformadas ou incorporadas."}, {"lei": "Art. 133, inciso I do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 integral e isolada, no caso de o alienante cessar a explora\u00e7\u00e3o da atividade."}, {"lei": "Art. 133, inciso II do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 subsidi\u00e1ria, se o alienante prosseguir na explora\u00e7\u00e3o da atividade ou, dentro de seis meses contados da data da aliena\u00e7\u00e3o, iniciar nova atividade no mesmo ou em outro ramo de com\u00e9rcio, ind\u00fastria ou profiss\u00e3o."}], "principio_geral_do_direito": [], "outros": []}""", "conteudo")
get_leis(""" {"constituicao_federal": [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [], "codigo_tributario_nacional": [{"lei": "Art. 132 do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A pessoa jur\u00eddica de direito privado que resultar de fus\u00e3o, transforma\u00e7\u00e3o ou incorpora\u00e7\u00e3o de outra ou em outra \u00e9 respons\u00e1vel pelos tributos devidos at\u00e9 \u00e0 data do ato pelas pessoas jur\u00eddicas de direito privado fusionadas, transformadas ou incorporadas."}, {"lei": "Art. 133, inciso I do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 integral e isolada, no caso de o alienante cessar a explora\u00e7\u00e3o da atividade."}, {"lei": "Art. 133, inciso II do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 subsidi\u00e1ria, se o alienante prosseguir na explora\u00e7\u00e3o da atividade ou, dentro de seis meses contados da data da aliena\u00e7\u00e3o, iniciar nova atividade no mesmo ou em outro ramo de com\u00e9rcio, ind\u00fastria ou profiss\u00e3o."}], "principio_geral_do_direito": [], "outros": []}""", "nomes")


get_leis(""" {'constituicao_federal': [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [], "codigo_tributario_nacional": [{"lei": "Art. 132 do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A pessoa jur\u00eddica de direito privado que resultar de fus\u00e3o, transforma\u00e7\u00e3o ou incorpora\u00e7\u00e3o de outra ou em outra \u00e9 respons\u00e1vel pelos tributos devidos at\u00e9 \u00e0 data do ato pelas pessoas jur\u00eddicas de direito privado fusionadas, transformadas ou incorporadas."}, {"lei": "Art. 133, inciso I do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 integral e isolada, no caso de o alienante cessar a explora\u00e7\u00e3o da atividade."}, {"lei": "Art. 133, inciso II do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 subsidi\u00e1ria, se o alienante prosseguir na explora\u00e7\u00e3o da atividade ou, dentro de seis meses contados da data da aliena\u00e7\u00e3o, iniciar nova atividade no mesmo ou em outro ramo de com\u00e9rcio, ind\u00fastria ou profiss\u00e3o."}], "principio_geral_do_direito": [], "outros": []}""", "nomes")

get_leis(""" {'constituicao_federal': [], 'codigo_penal': [], 'codigo_processo_penal': [], 'clt': [], 'codigo_civil': [], 'codigo_processo_civil': [{'lei': "Medeni Usul Kanunu'nun 21. Maddesi, II. Fıkrası", 'conteudo': "yükümlülüğün Brezilya'da yerine getirilmesi gereken davaları işlemeye ve yargılamaya Brezilya yargı mercii yetkilidir."}], 'codigo_direito_consumidor': [], 'estatuto_crianca_adolescente': [], 'sumula_stj': [], 'estatuto_oab': [], 'codigo_tributario_nacional': [], 'principio_geral_do_direito': []}""", "nomes")
get_leis(""" {'constituicao_federal': [], 'codigo_penal': [], 'codigo_processo_penal': [], 'clt': [], 'codigo_civil': [], 'codigo_processo_civil': [{'lei': "Medeni Usul Kanunu'nun 21. Maddesi, II. Fıkrası", 'conteudo': "yükümlülüğün Brezilya'da yerine getirilmesi gereken davaları işlemeye ve yargılamaya Brezilya yargı mercii yetkilidir."}], 'codigo_direito_consumidor': [], 'estatuto_crianca_adolescente': [], 'sumula_stj': [], 'estatuto_oab': [], 'codigo_tributario_nacional': [], 'principio_geral_do_direito': [], 'outros': [{"lei": 'aaa', "conteudo": 'bbb'}] } """, "nomes")

get_leis(""" {'constituicao_federal': [], 'codigo_penal': [], 'codigo_processo_penal': [], 'clt': [], 'codigo_civil': [], 'codigo_processo_civil': [{'lei': 'Art. 554 do CPC/2015', 'conteudo': 'Zilyetlik davalarında değiştirilebilirlik (fungibilidade) özelliğini belirler; bu da bir zilyetlik davası yerine başka bir davanın açılmasının, yargıcın talebi incelemesine ve koşulları kanıtlanmış olana karşılık gelen yasal korumayı sağlamasına engel olmayacağı anlamına gelir.'}, {'lei': 'Art. 564, parágrafo único, do CPC/2015', 'conteudo': 'Ön gerekçelendirme emredildiğinde, cevap verme süresi, ihtiyati tedbir kararını veren veya reddeden tebliğden itibaren sayılır.'}, {'lei': 'Art. 555 do CPC/2015', 'conteudo': 'Zilyetlik talebiyle birlikte, zarar ve ziyanın tazmini, semerelerin tazmini ve yeni bir tecavüz veya gasp durumunda para cezası uygulanması gibi taleplerin birleştirilmesine izin verir.'}, {'lei': 'Art. 560 do CPC/2015', 'conteudo': 'Zilyetlik sahibinin tecavüz durumunda zilyetliğinde tutulma ve gasp durumunda zilyetliğin iadesi hakkına sahip olduğunu belirtir.'}], 'codigo_direito_consumidor': [], 'estatuto_crianca_adolescente': [], 'sumula_stj': [], 'estatuto_oab': [], 'codigo_tributario_nacional': [], 'principio_geral_do_direito': [], 'outros': []} """, "nomes")

get_leis(""" {'constituicao_federal': [{'lei': '1988 Federal Anayasasının 146. maddesi, III. fıkrası, "b" bendi', 'conteudo': 'Tamamlayıcı yasaya düşen görev, vergi mevzuatı konusunda genel normları, yani vergilerin ve türlerinin tanımını, vergi doğuran olayları, matrahları, mükellefleri ile vergi yükümlülüğü, tahakkuk, alacak, zamanaşımı ve hak düşürücü süre konularını belirlemektir.'}], 'codigo_penal': [], 'codigo_processo_penal': [], 'clt': [], 'codigo_civil': [], 'codigo_processo_civil': [], 'codigo_direito_consumidor': [], 'estatuto_crianca_adolescente': [], 'sumula_stj': [], 'estatuto_oab': [], 'codigo_tributario_nacional': [], 'principio_geral_do_direito': [], 'outros': []} """, "nomes")

get_leis(""" {'constituicao_federal': [{'lei': '1988 Federal Anayasası\'nın 146. maddesi, III. fıkrası, b bendi', 'conteudo': 'Tamamlayıcı yasaya düşen görev, vergi mevzuatı konusunda genel normları, yani vergilerin ve türlerinin tanımını, vergi doğuran olayları, matrahları, mükellefleri ile vergi yükümlülüğü, tahakkuk, alacak, zamanaşımı ve hak düşürücü süre konularını belirlemektir.'}], 'codigo_penal': [], 'codigo_processo_penal': [], 'clt': [], 'codigo_civil': [], 'codigo_processo_civil': [], 'codigo_direito_consumidor': [], 'estatuto_crianca_adolescente': [], 'sumula_stj': [], 'estatuto_oab': [], 'codigo_tributario_nacional': [], 'principio_geral_do_direito': [], 'outros': []} """, "nomes")

In [ ]:
import json

def build_prompt(prompt_text, question_body, row, irac):

    if "G" in irac:
        return prompt_text + "\n" + question_body
    
    if "F" in irac:
        prompt_text = prompt_text + f"\nVou te dar, como dica, a lista de fatos contida na questão:\n \"{row['Facts']}\""

    if "I" in irac:
        #print(f"Vou adicionar issue: {row['Issue']}")
        prompt_text = prompt_text + f"\nVou te dar, como dica, a controvérsia principal contida na questão:\n \"{row['Issue']}\""

    if "L" in irac:
        #print('rule', row['Rule'])
        rules = get_leis(row['Rule'], "nomes")
        #print('rule', rules)
        prompt_text = prompt_text + f"\nVou te dar, como dica, as regras e leis que você deve aplicar para resolver a questão:\n {rules}\""

    if "R" in irac:
        #print('rule', row['Rule'])
        rules = get_leis(row['Rule'], "conteudo")
        #print('rule', rules)
        prompt_text = prompt_text + f"\nVou te dar, como dica, as regras e leis que você deve aplicar para resolver a questão:\n {rules}\""

    if "A" in irac:
        prompt_text = prompt_text + f"\nVou te dar, como dica, a lógica da aplicação das regras e leis que você deve usar para resolver a questão:\n \"{row['Application']}\""

    if "C" in irac:
        prompt_text = prompt_text + f"\nVou te dar, como dica, a conclusão do caso legal para resolver a questão:\n \"{row['Conclusion']}\""

    return prompt_text + "\n" + question_body
    

In [ ]:
import pandas as pd
import time
from tqdm import tqdm
import os

def run_exam(oab_questions_df, prompt, model_name, N=1, firac=""):
    assert oab_questions_df['question_id'].is_unique
    
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(model_name)

    output_path = f"../data/processed/model-runs/{language}/{model_name}_answers_firac-{firac}.csv"

    if os.path.exists(output_path):
        previous_answers_df = pd.read_csv(output_path)
        previous_answers_df = previous_answers_df[previous_answers_df['alternativa'].notna() & 
                                          (previous_answers_df['alternativa'] != '')].copy()

        assert previous_answers_df['question_id'].is_unique
        already_processed_ids = set(previous_answers_df["question_id"])
    else:
        previous_answers_df = pd.DataFrame()
        already_processed_ids = set()

    total_questions = len(oab_questions_df)
    processed_so_far = len(already_processed_ids)
    remaining = total_questions - processed_so_far
    percent_done = (processed_so_far / total_questions) * 100

    #print(f"--- Progresso atual --- {firac}")
    print(f"Já processadas: {processed_so_far} de {total_questions} [{language}] ({percent_done:.1f}%)")
    # print(f"Faltam: {remaining} questões de {language}")

    remaining_df = oab_questions_df[
        (~oab_questions_df["question_id"].isin(already_processed_ids))
        & ~(oab_questions_df["materia"] == "DIREITO PENAL")
    ]

    if remaining_df.empty:
        print("✅ Todas as questões já foram processadas.")
        return

    assert remaining_df['question_id'].is_unique

    amostra_df = remaining_df.sample(
        n=min(N, len(remaining_df)), random_state=43, replace=False
    ).reset_index(drop=True)
    assert amostra_df['question_id'].is_unique

    new_answers_df = pd.DataFrame(columns=[
        "pdf_filename", "materia", "prova",
         "questao", "question_id", "enunciado",
        "A", "B", "C", "D",
        "resposta_completa", "alternativa", "correct", "response_time_seconds"
    ])

    for index, row in tqdm(amostra_df.iterrows(), total=len(amostra_df), desc=f"Executando questões {model_name}"):
        print(f"Respondendo {row['question_id']}...")

        question_body = f'''
        Questão: {row['enunciado']}

        A) {row['A']}
        B) {row['B']}
        C) {row['C']}
        D) {row['D']}
        '''

        full_prompt = build_prompt(prompt, question_body, row, firac)

        #print(full_prompt)

        # ⏱️ medir tempo de resposta do modelo
        start_time = time.time()
        response = model.generate_content(full_prompt)
        end_time = time.time()
        response_time = end_time - start_time

        resposta_texto = response.text
        answer = parse_answer(resposta_texto)

        new_row_df = pd.DataFrame([{
            "pdf_filename": row['pdf_filename'],
            "question_id": row['question_id'],
            "materia": row['materia'],
            "language": row['language'],
            "prova": row['prova'],
            "questao": row['questao'],
            "enunciado": row['enunciado'],
            "prompt": full_prompt,
            "A": row['A'],
            "B": row['B'],
            "C": row['C'],
            "D": row['D'],
            "resposta_completa": resposta_texto,
            "alternativa": answer,
            "firac": firac,
            "rule_count": row['rule_count'],
            "correct": row['correct'],
            "response_time_seconds": response_time  
        }])
    # Filtra dataframes vazios antes da concatenação
    frames = [f for f in [new_answers_df, new_row_df] if not f.empty and not f.isna().all().all()]
    new_answers_df = pd.concat(frames, ignore_index=True) if frames else new_row_df.copy()

    time.sleep(5)

    frames = [previous_answers_df, new_answers_df]
    # Mantém apenas DataFrames que não são vazios e não têm todas as colunas NaN
    frames = [f for f in frames if not f.empty and not f.isna().all().all()]
    updated_answers_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    
    assert updated_answers_df['question_id'].is_unique

    updated_answers_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    processed_total = len(updated_answers_df)
    percent_final = (processed_total / total_questions) * 100
    print(f"\n✅ Processadas {len(new_answers_df)} novas questões.")
    print(f"📊 Total agora: {processed_total} de {total_questions} ({percent_final:.1f}%)")


In [ ]:
import random
import time

# Início da contagem de tempo
start_time = time.time()

N = 200

firacs = ["____", "__L__", "F____", "__R__", "FI___", "FIL__", "FIR__", "FIRA_", "FIRAC"]
# models = ["gemini-1.5-flash-8b", "gemini-1.5-flash", "gemini-2.0-flash-lite", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash", "gemini-2.5-pro"]
models = ["gemma-3-1b-it", "gemma-3-4b-it", "gemma-3-12b-it", "gemma-3-27b-it"] #"gemini-2.0-flash-lite", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-2.5-pro"]

random.shuffle(firacs)
random.shuffle(models)

runs = 0
total_runs = len(firacs) * len(models) * N

for i in range(N):
    for firac in firacs:
        print(f"\n**{firac} **")
        for model in models:
            runs += 1
            print(f"run {runs}/{total_runs}")
            # try:
            run_exam(oab_questions_df, prompt_text_question, model, firac=firac)
            # except Exception as e:
            #     print(f"Erro for model={model}, firac={firac}: {e}")
            #     time.sleep(60)
            #     continue

# Fim da contagem de tempo
end_time = time.time()
elapsed = end_time - start_time

# Exibir tempo total formatado (em horas, minutos e segundos)
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)

print(f"\nTempo total de execução: {hours}h {minutes}m {seconds}s ({elapsed:.2f} segundos)")


In [ ]:
'''
import random

N = 10


firacs = ["G"]
models = ["gemini-1.5-flash-8b", "gemini-1.5-flash", "gemini-2.0-flash-lite", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash", "gemini-2.5-pro"]
models = ["gemini-2.0-flash-lite", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash"]


random.shuffle(firacs)
random.shuffle(models)

for i in range(0, N):
    for firac in firacs:
        print("**", firac, "**")
        for model in models:
            run_exam(oab_guess_the_rule_df, prompt_guess_rule, model, firac=firac)

'''








